In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
#define global variables
##scratch directory 
##work directory
##work1 ; directory for file from past experiment

data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_get_cohort_overlap_patient_count"
results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-13_get_cohort_overlap_patient_count" 
scratch = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-11-13_get_cohort_overlap_patient_count"

results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-13_get_conditions_of_cohorts" 
#!mkdir {scratch}

In [ ]:
## 1. import data pkl

In [ ]:
def get_cohort_dict_pkl():

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{data}/all_viral_disease_cohort_conditions_df.pkl')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
## 2. Wrangle data

In [ ]:
# --- NEW CODE TO CREATE THE DICTIONARY ---

def get_cond_cohort_df_dict(all_cond_df): 
    print("Grouping DataFrame into a dictionary...")

    # Define the columns you want to use for your composite key
    key_columns = ['condition_concept_id', 'standard_concept_name']

    # Use a dictionary comprehension with groupby to create the dictionary
    # - The 'key' will be a tuple: (condition_concept_id, standard_concept_name)
    # - The 'group_df' will be the DataFrame containing all rows for that key
    concept_groups_dict = {
        key: group_df 
        for key, group_df in all_cond_df.groupby(key_columns)
    }

    print(f"Successfully created a dictionary with {len(concept_groups_dict)} unique (ID, Name) keys.")
    
    return concept_groups_dict

In [ ]:
def filter_cond_df_dict(all_cond_dict):
    
    master_df_of_final_cohorts = pd.read_csv(f"{results1}/viral_disease_condition_cohorts.csv")

    master_key_cols = ['condition_concept_id', 'standard_concept_name']

    # Create a set of tuples (key1, key2) from the master DataFrame.
    # This set will act as our "allow list".
    valid_keys_set = set(
        master_df_of_final_cohorts[master_key_cols].itertuples(index=False, name=None)
    )
    
    # 'concept_groups_dict' is the dictionary you created in the previous step
    # 'valid_keys_set' is the set we just created

    filtered_concept_dict = {
        key: group_df 
        for key, group_df in all_cond_dict.items() 
        if key in valid_keys_set
    }

    print(f"Original dictionary had {len(all_cond_dict)} items.")
    print(f"Filtered dictionary now has {len(filtered_concept_dict)} items.")

    # You can now work with your new, smaller dictionary
    # print(list(filtered_concept_dict.keys())[:5])
    
    return filtered_concept_dict

In [ ]:
## 3. viral overlap

In [ ]:
viral_overlap = dict()
    
def viral_overlap_dict(viral_cohort_dict):
    
    
    for key, table in viral_cohort_dict.items():
        
        person_IDs = table['person_id']
        
        concept_names_series = table['standard_concept_name']
        concept_names_str = key[1]
    
        viral_overlap[concept_names_str]= set(person_IDs)
        
        
    return viral_overlap

In [ ]:
def viral_overlap_count(viral_overlap):
    
    max_overlap = []
    all_concept_names = viral_overlap.keys() #a list of dictionary keys (concept_names)
    less_than_20 = set()
    
    
    for concept_name in all_concept_names: #concept_name = concept name per row for each item in all_cocnept_names list
        max_concept_name = None
        overlap = 0
        person_ids = viral_overlap[concept_name] #list of person_id asscoiated with each key in viral_overlap dict
        person_ids_count = len(person_ids) #count of person_id asscociated with each key

        
        #initialize in case no overlap is found
        max_concept_count = 0
        overlap_percent = 0
        max_concept_overlap_percent = 0
        average = 0
        
        
        
        for concept_name_to_check in all_concept_names: #all concept names =  all the concept names again for each concept name indivdually in a list
                                                        #concept_name_to_check = each of those concept names individually from the list of concept_names used to check for each original key 

            person_ids_to_check = viral_overlap[concept_name_to_check] #person_ids associated with each concept_name , in the list of concept names to check for each original key
            number_of_overlaps = len(person_ids.intersection(person_ids_to_check)) #the overlap of those person ids, length of list of intersection of two list
            

        
            
            if person_ids == person_ids_to_check: #skipping self comparison (logical error)
                continue
            
            
        
            if number_of_overlaps > overlap:  #number_of_overlaps = if greater than 0, then overlap now = # of overlaps #upates the variable 
               
                overlap = number_of_overlaps #number of overlaps between each concept name where the # of overlaps is the greatest
                max_concept_name = concept_name_to_check #concep_name where the # of overlaps is the greatest 
            
                overlap_percent = int((overlap/person_ids_count)*100)  # (number of max_overlap/ person_id count )*100
                max_concept_count = len(viral_overlap[max_concept_name]) #count of person_id in the max_concept_name
                max_concept_overlap_percent = int((overlap/max_concept_count)*100) # (number of max_overlap/ max_concept person_id count )*100
                average = int((overlap_percent + max_concept_overlap_percent)/2) # (overlap_percent + max_concept_overlap_percent) / 2
      
                
        max_overlap.append([concept_name, person_ids_count, max_concept_name, max_concept_count, overlap, overlap_percent, max_concept_overlap_percent, average])
       
        
        final_df = pd.DataFrame(max_overlap)
        final_df.columns = ['standard_concept_name', 'count', 'max_concept_name', 'max_concept_count', 'overlap_count', 'overlap_percent', 'max_concept_overlap_percent', 'average']
        
    
    #add concept_ids
    
    final_cohort_concept_id = pd.read_csv(f"{results1}/viral_disease_condition_cohorts.csv")
    
    concept_id = list(final_cohort_concept_id['condition_concept_id'])
    
    final_df['condition_concept_id'] = concept_id
    
    final_df = final_df[['condition_concept_id', 'standard_concept_name', 'count', 'max_concept_name',
    'max_concept_count', 'overlap_count', 'overlap_percent',
    'max_concept_overlap_percent', 'average',]]
        
       
    df = final_df.sort_values(by='count', ascending=False)

    return df


In [ ]:
##function calls

all_cond_df = get_cohort_dict_pkl()

all_cond_dict = get_cond_cohort_df_dict(all_cond_df)

filtered_cond_dict = filter_cond_df_dict(all_cond_dict)



person_id_overlap_dict = viral_overlap_dict(filtered_cond_dict)

viral_cohort_overlap_count = viral_overlap_count(person_id_overlap_dict)

In [ ]:
#all_cond_df
#all_cond_dict
#filtered_cond_dict

#person_id_overlap_dict

viral_cohort_overlap_count

In [ ]:
viral_cohort_overlap_count.to_csv(f"{scratch}/cohort_overlap_count.csv")